In [1]:
%pip install geopy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from geopy.geocoders import OpenCage
from geopy.extra.rate_limiter import RateLimiter
from tqdm.notebook import tqdm

API_KEY = "949b2a01a76e4d98ae7f03a5cfb26daa"
ADDRESS_COLUMN = "address"
COUNTRY = "Netherlands"
INPUT_FILE = "clients.csv"
OUTPUT_FILE = "clients_with_coords.csv"

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows.")
display(df.head())

geolocator = OpenCage(api_key=API_KEY, timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.0)

def get_lat_lon(addr):
    if pd.isna(addr) or not addr:
        return None, None
    try:
        location = geocode(f"{addr}, {COUNTRY}")
        if location:
            return location.latitude, location.longitude
    except Exception as e:
        print(f"Error: {addr} -> {e}")
    return None, None

tqdm.pandas(desc="Geocoding")
df["latitude"], df["longitude"] = zip(*df[ADDRESS_COLUMN].progress_apply(get_lat_lon))

success = df[["latitude", "longitude"]].notna().all(axis=1).sum()
print(f"Success: {success}/{len(df)} = {success/len(df):.1%}")
display(df.head())

df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to {OUTPUT_FILE}")

Loaded 100 rows.


,name,address,care_arrangement,preferences,time_window_start,time_window_end,care_hours,dogs,cats,smokes
0,Client 1,A gen Giezen 10 6418 OY Heerlen,HBH Basic,afternoon,12:00,18:00,2.5,0,1,False
1,Client 2,Caumerweg 30 6418 IW Heerlen,HBH Basic,afternoon,12:00,18:00,3.0,2,0,False
2,Client 3,Kortstraat 7 6411 CW Heerlen,V&V,afternoon,12:00,18:00,1.5,0,2,False
3,Client 4,Oliemolenstraat 16 6411 EQ Heerlen,HBH Plus,afternoon,12:00,18:00,1.0,0,0,False
4,Client 5,September 1944-straat 9 6418 ZV Heerlen,HBH Plus,afternoon,12:00,18:00,2.5,0,0,True


Geocoding:   0%|          | 0/100 [00:00<?, ?it/s]

Success: 100/100 = 100.0%


,name,address,care_arrangement,preferences,time_window_start,time_window_end,care_hours,dogs,cats,smokes,latitude,longitude
0,Client 1,A gen Giezen 10 6418 OY Heerlen,HBH Basic,afternoon,12:00,18:00,2.5,0,1,False,50.88365,5.98154
1,Client 2,Caumerweg 30 6418 IW Heerlen,HBH Basic,afternoon,12:00,18:00,3.0,2,0,False,50.88365,5.98154
2,Client 3,Kortstraat 7 6411 CW Heerlen,V&V,afternoon,12:00,18:00,1.5,0,2,False,50.88365,5.98154
3,Client 4,Oliemolenstraat 16 6411 EQ Heerlen,HBH Plus,afternoon,12:00,18:00,1.0,0,0,False,50.88365,5.98154
4,Client 5,September 1944-straat 9 6418 ZV Heerlen,HBH Plus,afternoon,12:00,18:00,2.5,0,0,True,50.88365,5.98154


Saved to clients_with_coords.csv
